In [ ]:
!pip install -q aiohttp wasmtime nest_asyncio yt-dlp
!pip install -q faster-whisper
!pip install -q git+https://github.com/sos-pc/scrapower.git

# -- CUDA diagnostic (run BEFORE worker, captures env state) --
import subprocess, os, sys
print("=== NVIDIA-SMI ===")
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout[-600:] if r.stdout else "nvidia-smi failed")

print("\n=== PyTorch CUDA ===")
try:
    import torch
    print(f"torch {torch.__version__}, CUDA {torch.version.cuda}")
    print(f"cuDNN bundled: {torch.backends.cudnn.version()}")
    print(f"CUDA available: {torch.cuda.is_available()}, GPU count: {torch.cuda.device_count()}")
except ImportError:
    print("torch not installed")

print("\n=== LD_LIBRARY_PATH ===")
print(os.environ.get("LD_LIBRARY_PATH", "(empty)"))

print("\n=== /usr/local/cuda ===")
for cuda_dir in ["/usr/local/cuda", "/usr/local/cuda-12"]:
    if os.path.isdir(cuda_dir):
        libs = [f for f in os.listdir(f"{cuda_dir}/lib64") if 'cudnn' in f.lower()] if os.path.isdir(f"{cuda_dir}/lib64") else []
        print(f"{cuda_dir}: lib64 cudnn libs: {libs[:5]}")

print("\n=== ctranslate2 ===")
try:
    import ctranslate2
    print(f"version: {ctranslate2.__version__}")
    print(f"CUDA devices: {ctranslate2.get_cuda_device_count()}")
except ImportError as e:
    print(f"not installed: {e}")
except Exception as e:
    print(f"ERROR: {e}")


In [ ]:
import os
import nest_asyncio; nest_asyncio.apply()

os.environ["IDLE_TIMEOUT_SEC"] = "60"
os.environ["COORDINATOR_URL"] = "{{COORDINATOR_URL}}"
os.environ["SCRAPOWER_API_KEY"] = "{{API_KEY}}"
os.environ["SCRAPOWER_WORKER_PREFIX"] = "kaggle"
# WG_PROXY assembled at runtime from injected components
_user = "{{WG_USER}}"
_pass = "{{WG_PASS}}"
_host = "{{WG_HOST}}"
if _user and _pass and _host:
    os.environ["WG_PROXY"] = f"socks5://{_user}:{_pass}@{_host}:1081"

from scrapower.worker.entry import main
main()
